# 🎙️ تشغيل وتجربة QwenCleo-ASR على كروت الشاشة المجانية في Kaggle (Dual T4 GPUs)

هذه المفكرة مجهزة بالكامل لتفريغ المحاضرات المصرية الطويلة والمصطلحات التقنية بأعلى دقة وسرعة باستخدام **2x Nvidia T4 GPUs**.

### ⚙️ الخطوة الأولى في كاجل:
1. من القائمة الجانبية يمين الصفحة **Notebook options**:
   - **Accelerator**: اختر **GPU T4 x2** (أو GPU P100)
   - **Internet**: فعّل **Internet ON** (ضروري لتحميل الحزم والنموذج)

In [ ]:
# 1. التحقق من كروت الشاشة وتثبيت المكتبات المطلوبة
!nvidia-smi
!pip install -q qwencleo-asr ffmpeg-python soundfile jiwer tqdm

In [ ]:
# 2. استيراد المكتبات والتأكد من تفعيل CUDA
import os, sys, time, json
import torch

print(f"🔥 CUDA Available: {torch.cuda.is_available()}")
print(f"🔥 GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"   - GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB VRAM)")

In [ ]:
# 3. تحميل نموذج QwenCleo-ASR على كارت الشاشة الأول
from qwencleo_asr import QwenCleoASR

print("⏳ جارٍ تحميل نموذج QwenCleo-ASR على GPU...")
t0 = time.time()
asr = QwenCleoASR(
    model_id="mohammedaly22/QwenCleo-ASR",
    device="cuda:0",
    dtype="bfloat16",
    quiet=False
)
print(f"✅ تم تحميل النموذج بنجاح في {time.time() - t0:.2f} ثانية!")

In [ ]:
# 4. دالة التفريغ واستخراج الصوت وتصدير SRT و TXT
def transcribe_lecture(video_or_audio_path, output_name="lecture_transcript"):
    wav_path = f"/tmp/{output_name}_16k.wav"
    print(f"🎵 استخراج الصوت من {video_or_audio_path}...")
    os.system(f'ffmpeg -y -i "{video_or_audio_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{wav_path}" -loglevel error')
    
    print("🎙️ بدء تفريغ الصوت بنموذج QwenCleo-ASR...")
    t_start = time.time()
    res = asr.transcribe(wav_path)
    elapsed = time.time() - t_start
    
    txt_file = f"{output_name}.txt"
    with open(txt_file, "w", encoding="utf-8") as f:
        f.write(res.text)
        
    print("=" * 60)
    print(f"✅ اكتمل التفريغ في {elapsed:.2f} ثانية ({elapsed/60.0:.2f} دقيقة)!")
    print(f"📄 تم حفظ النص في: {txt_file}")
    print("=" * 60)
    
    print("\n--- عينة من أول النص المفرغ ---")
    print(res.text[:500])
    return res

In [ ]:
# 5. تجربة تفريغ ملف (ضع مسار ملفك هنا أو ارفعه في Kaggle)
# مثال: يمكنك رفع الملف من زر Upload في Kaggle أو وضع رابط مباشر
# transcribe_lecture("/kaggle/working/your_video.mp4", "my_lecture")